# Jun-2026 LAPD ion-saturation current (Isat) — fluctuation explorer

Companion to `Jun2026_IV_explore.ipynb`.  That notebook walks the *swept*
Langmuir tips (complete I+V pairs) to extract Vp/Te/ne; this one reads a
**fixed-bias ion-saturation channel** and looks at how the Isat signal
*fluctuates* in time (raw trace + FFT).

Reader + FFT helpers live in `Jun2026_Isat.py` (which reuses the read path and
knobs from `Jun2026_IV.py`); this notebook just drives them.  **You pick the
Isat channel by hand** — the notebook prints every channel's description and the
run's probe settings, then you name the scope + channel.

**This stage saves nothing** — no files, no batch loop.

Workflow: configure → list channel descriptions + run setup → select channel →
positions → read raw signal + plot. *(Pause here — FFT / fluctuation analysis
comes next.)*

## 1. Configure — set the run file

In [1]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import numpy as np
import matplotlib.pyplot as plt

import Jun2026_IV as jiv
import Jun2026_Isat as jis
importlib.reload(jiv)
importlib.reload(jis)   # re-run this cell after editing Jun2026_Isat.py / Jun2026_IV.py

from data_analysis.io import open_lapd

# >>> SET THIS to the run you want to inspect <<<
# (a run with fixed-bias Isat channels, e.g. the 30-/32- 'Isat, ...' runs)
ifn = r"D:\data\LAPD\jun2026-jia\07-He-800G-bias40V-Isat-p29-plane_2026-06-10.hdf5"

# Current scaling knobs live in Jun2026_IV.py and are reused here (RESISTOR,
# Aprobe).  For fluctuation work the *shape* of the spectrum matters, not the
# absolute scaling.  The IV pipeline's I_SIGN is irrelevant to Isat and is not
# applied.
print(f"RESISTOR = {jiv.RESISTOR}, Aprobe = {jiv.Aprobe}")

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)

RESISTOR = 25.0, Aprobe = 0.002
backend: pydaq


## 2. List channel descriptions + run setup

Print every scope group's channel descriptions and the run's hand-written
description (plasma / bias / probe settings).  Read these to decide which scope +
channel is the Isat channel you want — the selection is made by hand in the next
cell, nothing is guessed here.

In [2]:
channels = jis.list_all_channels(ifn)
print()
jis.print_run_description(ifn);

Scope groups and channel descriptions:

  scope 'machscope':
    C2: 'MP@P33, Isat-Y+'
    C3: 'LP@P29, Isat-L'
    C4: 'LP@P29, Isat-R'

=== Run description ===
Documenting instability observed during electrode biasing with the main purpose of cross-correlation
Langmuir Isat plane with moving probe at P29 and reference probe at P33 (0,-4)

Operator: Jia Han, PP, Zoe
    
Setup:
    - Plasma condition: 
      - Heater 1860A,  bank 90V, anode-cathode 50V, current 4kA
      - Helium backside pressure 40 Psi, Puff voltage 75V for 32ms West+East
      - Discharge 20 ms, Pulsing 1/3 Hz, plasma breakdown ~10 ms
      - Pressure unknown
      - Port 20, density at 10 ms:  0.7e13 cm^-3
      - Port 29, density at 10 ms:  1e13 cm^-3
    - Magnetic field
      - Black magnets at south: 888 A (PS12, 13),
      - Yellow & Purple magnets: configured for uniform 800 G
      - Black magnets at north: 0 A (PS11)
    - Multi-electrode bias: Port 35 top, centered ~(0,0)
      - Five circular disc shaped

## 3. Select the Isat channel

From the lists above, set the scope group and channel of the Isat signal you
want to inspect.

In [10]:
# >>> SET THESE to the Isat channel you want (from the lists above) <<<
scope_name = "machscope"
chan       = "C2"

desc = channels.get(scope_name, {}).get(chan, "(no description)")
print(f"Selected Isat channel: scope '{scope_name}' {chan}  ->  {desc!r}")
assert scope_name in channels, f"scope {scope_name!r} not in {list(channels)}"
assert chan in channels[scope_name], f"channel {chan!r} not in {list(channels[scope_name])}"

Selected Isat channel: scope 'machscope' C2  ->  'MP@P33, Isat-Y+'


## 4. Positions — pick one to inspect

In [11]:
# If the run has more than one probe drive (multiple motion groups), set
# motion_group_name to the one whose probe carries the Isat channel above
# (read_lp_positions lists the groups in its error). None works for single-drive runs.
motion_group_name = None    # e.g. "<Hermes>    p29_LP"

pos_array, xpos, ypos, npos, nshot = jiv.read_lp_positions(ifn, motion_group_name)

pos_index = npos//2     # <<< which position to inspect (0 .. npos-1)
print(f"\nInspecting position index {pos_index} of {npos} "
      f"(x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), {nshot} shots")

Using motion group: '<Hermes>    p29_LP'
Positions: 1369 unique (x: 37, y: 37), 5 shots/position, 6845 total shots.

Inspecting position index 684 of 1369 (x=? cm), 5 shots


## 5. Read raw Isat signal at this position + plot

Per-shot Isat traces (scaled per the knobs) at the chosen position.  Inspect the
raw signal: the mean level is the ion-saturation current; the scatter about it
shot-to-shot and the wiggle within a shot are the fluctuations we'll FFT next.

In [12]:
# uses scope_name / chan selected in section 3
tarr, Iarr = jis.get_isat_at_position(run, scope_name, chan, npos, nshot, pos_index)
print("Iarr:", Iarr.shape, " tarr:", tarr.shape)

fig, ax = plt.subplots()
for s in range(Iarr.shape[0]):
    ax.plot(tarr * 1e3, Iarr[s], lw=0.6, alpha=0.5)
ax.plot(tarr * 1e3, Iarr.mean(0), "k", lw=1.2, label="shot mean")
ax.set_xlabel("t [ms]"); ax.set_ylabel("Isat [a.u.]")
ax.set_title(f"Raw Isat — pos {pos_index} (x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), "
             f"scope '{scope_name}' {chan}: {desc!r}")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

Iarr: (5, 100001)  tarr: (100001,)


## 6. Downsampling options — compare on 1 shot at 3 positions

The raw Isat trace is ~2.5M samples/shot, which makes plotting (and any
interactive figure carrying it) slow. Before the FFT stage, decide how to
**downsample** for display.

Three options, each reducing the sample count by the same factor `Q`:

1. **Stride** — `I[::Q]`: keep every Q-th sample. Cheapest, but it *aliases*:
   fluctuations above the new Nyquist fold back into the band. Fine for a quick
   look at the slow envelope, misleading for spectral content.
2. **Block mean** — average each non-overlapping block of `Q` samples. A crude
   boxcar low-pass + decimate: anti-aliases somewhat and is robust, so it tracks
   the *mean level / slow structure* well, but the boxcar has poor stopband
   rejection.
3. **`scipy.signal.decimate`** — polyphase FIR-filtered decimation: proper
   anti-aliasing filter then downsample. The principled choice when the
   downsampled trace will be FFT'd; preserves in-band fluctuations without
   aliasing.

Below: one shot (`shot_sel`) at 3 positions, raw trace overlaid with all three
options at factor `Q`. Read these to pick the option + factor for the FFT stage.

In [ ]:
from data_analysis.signal.core import (
    downsample_stride, downsample_blockmean, downsample_decimate,
    DOWNSAMPLE_METHODS,
)

# --- downsampling factor + which single shot to show ---
Q        = 100   # decimation factor (same for all three options, so they're comparable)
shot_sel = 0     # which shot (0 .. nshot-1) at each position
TMIN_MS, TMAX_MS = 0.0, 10.0   # display window (ms) -- focus on 0..10 ms for now

# Three positions to compare (stationary probe, so these should look alike;
# differences are shot-to-shot / drift, not spatial). Clip to valid range.
pos_sel = [npos // 4, npos // 2, 3 * npos // 4]
pos_sel = sorted({min(max(p, 0), npos - 1) for p in pos_sel})

# Downsamplers are generic (data_analysis.signal.core); colors are a display
# choice, so pair them here. Same names/order as DOWNSAMPLE_METHODS.
_colors = {"stride": "tab:orange", "block mean": "tab:green", "decimate(FIR)": "tab:red"}
_options = [(name, DOWNSAMPLE_METHODS[name], _colors[name]) for name in DOWNSAMPLE_METHODS]

fig, axes = plt.subplots(len(pos_sel), 1, figsize=(12, 3.2 * len(pos_sel)),
                         sharex=True)
axes = np.atleast_1d(axes)

for ax, pidx in zip(axes, pos_sel):
    tarr_p, Iarr_p = jis.get_isat_at_position(run, scope_name, chan, npos, nshot, pidx)
    I = Iarr_p[shot_sel]
    t_ms = tarr_p * 1e3

    ax.plot(t_ms, I, color="0.6", lw=0.5, label=f"raw ({I.size:,} pts)")
    for name, fn, c in _options:
        td, xd = fn(tarr_p, I, Q)
        ax.plot(td * 1e3, xd, color=c, lw=1.0, alpha=0.9,
                label=f"{name} (Q={Q}, {xd.size:,} pts)")

    xcm = xpos[pidx] if pidx < len(xpos) else "?"
    ax.set_title(f"pos {pidx} (x={xcm} cm), shot {shot_sel}", fontsize=10)
    ax.set_ylabel("Isat [a.u.]")
    ax.set_xlim(TMIN_MS, TMAX_MS)   # focus on 0..10 ms
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=8, ncol=2)

axes[-1].set_xlabel("t [ms]")
fig.suptitle(f"Isat downsampling options (factor Q={Q})  —  "
             f"scope '{scope_name}' {chan}: {desc!r}", fontsize=11)
plt.tight_layout()
plt.show()


## 7. FFT of the Isat fluctuations over a chosen time window

Now that you've looked at the raw trace, pick the time window `[t_start, t_stop]`
(in **ms**) where the fluctuation you care about lives.  The FFT runs on the
**raw** windowed trace (decimating first makes the averaged spectrum noticeably
rougher).

1. searches the time array for the indices bracketing `t_start` / `t_stop`,
2. FFTs each shot's raw trace **trimmed to that window** (`amplitude_spectrum`
   detrends, so DC is removed and the window length sets the resolution
   `df = 1 / (t_stop - t_start)`),
3. averages the per-shot amplitude spectra over `n_shots`,
4. plots per-shot spectra (faint) + the averaged spectrum (full band).

Averaging is done on the **amplitude spectra** (incoherent average): random
shot-to-shot phase cancels in a coherent average but not here, so broadband
fluctuation power survives.

In [13]:
# >>> SET the FFT time window (ms), read off the raw trace in section 5 <<<
from data_analysis.signal import amplitude_spectrum

t_start, t_stop = 1.0, 5.0     # window for the FFT (ms)
n_shots = nshot                # how many shots to average (<= nshot)

# Read this position's shots (section 5 uses the same call; re-read so this cell
# is self-contained and `pos_index` can be changed independently above).
tarr, Iarr = jis.get_isat_at_position(run, scope_name, chan, npos, nshot, pos_index)
t_ms = tarr * 1e3
dt = tarr[1] - tarr[0]         # sample spacing (s) -> FFT frequency axis

# Find the indices bracketing the window. searchsorted on the ascending time
# array; clip n_shots to what's available.
i0, i1 = np.searchsorted(t_ms, [t_start, t_stop])
i0, i1 = int(i0), int(min(i1, t_ms.size))
n_shots = min(n_shots, Iarr.shape[0])
assert i1 - i0 >= 2, f"window [{t_start}, {t_stop}] ms selects < 2 samples (i0={i0}, i1={i1})"
print(f"window [{t_start}, {t_stop}] ms -> samples [{i0}:{i1}] "
      f"({i1 - i0:,} pts), averaging {n_shots} shots")

# FFT each shot's RAW windowed trace, then incoherently average the amplitude
# spectra. (FFT on the raw signal -- decimating first makes the spectrum rough.)
# amplitude_spectrum is the shared single-trace DSP; pass dt directly. detrend
# removes the mean so DC doesn't swamp the fluctuation spectrum.
amps = []
for s in range(n_shots):
    freq, amp = amplitude_spectrum(Iarr[s, i0:i1], dt, detrend=True)
    amps.append(amp)
amps = np.array(amps)
amp_mean = amps.mean(axis=0)

fig, ax = plt.subplots()
for s in range(n_shots):
    ax.semilogy(freq * 1e-3, amps[s], color="0.7", lw=0.5, alpha=0.6)
ax.semilogy(freq * 1e-3, amp_mean, "k", lw=1.3, label=f"mean of {n_shots} shots")
ax.set_xlabel("frequency [kHz]"); ax.set_ylabel("Isat amplitude [a.u.]")
ax.set_title(f"Isat FFT — pos {pos_index} "
             f"(x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), "
             f"window {t_start}-{t_stop} ms, scope '{scope_name}' {chan}: {desc!r}")
ax.set_xlim(left=0)
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()


window [1.0, 5.0] ms -> samples [35001:55001] (20,000 pts), averaging 5 shots


## 8. Batch results — averaged FFT for every run (00-06)

Sections 5-7 explore one run interactively. The batch routine
`jis.batch_fft()` (run it once, e.g. `python Jun2026_Isat.py`) does the same
all-shot-averaged FFT for **every** run 00-06 — the P33 Isat probe is stationary
across those runs, so it averages over *all* shots in each file — and writes one
npz (`isat_fft_00-06.npz`) into the data dir.

This cell loads that npz and plots each run's averaged spectrum in its own
panel of a single figure, so the runs can be compared side by side. The scope /
channel / window came from the constants at the top of `Jun2026_Isat.py`
(`SCOPE_NAME`, `CHAN`, `FFT_TMIN_MS`, `FFT_TMAX_MS`) — edit them there and re-run
`batch_fft()` to regenerate.

In [ ]:
import os

# npz written by jis.batch_fft() -- one shared `freq`, one `<run>__amp` per run.
npz_path = os.path.join(jis.DATA_DIR, jis.OUT_NPZ)
d = np.load(npz_path)

freq = d["freq"]
runs     = [str(r) for r in d["runs"]]
nshots   = d["nshots"]

# The fluctuation power lives in the low-frequency band; the rest is flat noise
# floor out to the ~25 MHz Nyquist. Zoom the x-axis there so the structure is
# readable. Set FMAX_KHZ = None to show the full band instead.
FMAX_KHZ = 100.0

# One panel per run, shared axes so the spectra are directly comparable.
fig, axes = plt.subplots(len(runs), 1, figsize=(11, 2.0 * len(runs)), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, run_key, n in zip(axes, runs, nshots):
    amp = d[f"{run_key}__amp"]
    ax.semilogy(freq *1e-3, amp, lw=0.8)
    # Run key starts with the 2-digit run number, e.g. "03-He-800G-...".
    ax.set_ylabel(run_key.split("-")[0], rotation=0, ha="right", va="center", fontsize=11)
    ax.grid(True, which="both", alpha=0.3)

axes[-1].set_xlabel("frequency [kHz]")
axes[-1].set_xlim(0, FMAX_KHZ)
fig.suptitle(f"Isat averaged FFT, runs 00-06  —  scope '{jis.SCOPE_NAME}' "
             f"{jis.CHAN}, window {jis.FFT_TMIN_MS}-{jis.FFT_TMAX_MS} ms",
             fontsize=11)
fig.supylabel("Isat amplitude [a.u.]")
plt.tight_layout()
plt.show()


---
**Pause here.** Raw Isat signal read and plotted for one position/channel.
Next step: FFT the fluctuations (per-shot + averaged spectra) — to be added.